In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras import Sequential, layers, optimizers, callbacks
import seaborn as sns
from tensorflow import keras

In [2]:
# Charger CIFAR-10
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Créer des datasets Tensorflow
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train.flatten()))
val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test.flatten()))

class_names = ['avion', 'automobile', 'oiseau', 'chat', 'cerf', 'chien', 'grenouille', 'cheval', 'bateau', 'camion']

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Pixel range: [{X_train.min()}, {X_train.max()}]")

X_train shape: (50000, 32, 32, 3)
y_train shape: (50000, 1)
X_test shape: (10000, 32, 32, 3)
y_test shape: (10000, 1)
Pixel range: [0, 255]


In [3]:
def preprocess(image, label):
    image = tf.image.resize(image, (224, 224))
    image = image / 255.0

    return image, label


# Pipeline optimisé
train_ds_pipeline = (train_ds
                     .map(preprocess)
                     .cache() # évite de recharger les données à chaque epoch
                     .shuffle(1000) # Mélange les données
                     .batch(64)
                     .prefetch(tf.data.AUTOTUNE)) # précharge les batchs suivants


# Pipeline optimisé
val_ds_pipeline = (val_ds
                     .map(preprocess)
                     .cache() # évite de recharger les données à chaque epoch
                     .batch(64)
                     .prefetch(tf.data.AUTOTUNE)) # précharge les batchs suivants

In [4]:
base_model = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False, # pas de couche de classification
    weights="imagenet", # Poids pré-entrainés
    alpha=0.75 # Version plus légère (75% de la taille)
)

# Geler toutes les couches du modèle de base
base_model.trainable = False

print(base_model.count_params())

1382064


In [5]:
model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(), # Pooling global
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_0.75_224            │ (None, 7, 7, 1280)     │     1,382,064 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,547,322 (5.90 MB)

 Trainable params: 165,258 (645.54 KB)

 Non-trainable params: 1,382,064 (5.27 MB)

In [6]:
history = model.fit(
    train_ds_pipeline,
    validation_data=val_ds_pipeline,
    epochs=3,
    verbose=1
)

Epoch 1/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 253s 320ms/step - accuracy: 0.7409 - loss: 0.7446 - val_accuracy: 0.7801 - val_loss: 0.6430
Epoch 2/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 247s 316ms/step - accuracy: 0.7950 - loss: 0.5862 - val_accuracy: 0.7802 - val_loss: 0.6292
Epoch 3/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 250s 320ms/step - accuracy: 0.8120 - loss: 0.5361 - val_accuracy: 0.7959 - val_loss: 0.5893


In [ ]:
model_scratch = keras.Sequential(
    [
        layers.Conv2D(32, 3, activation="relu", input_shape=(224,224,3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),

        layers.Conv2D(128, 3, activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(10, activation="softmax")
    ]
)

model_scratch.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_scratch = model_scratch.fit(
    train_ds_pipeline,
    validation_data=val_ds_pipeline,
    epochs=3,
    verbose=1
)

c:\Users\Administrateur\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/3


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 with name 'None' of layer 'dense_2' is incompatible with the layer: expected axis -1 of input shape to have value 512, but received input with shape (None, 86528)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(None, 224, 224, 3), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>